In [1]:
!pip install rank-bm25 openai tiktoken tqdm

In [2]:
import json
import os
import sys
import hashlib
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / "data" / "raw"
EVAL_DIR   = ROOT / "eval" / "golden_dataset" / "docs"
EVAL_FILE  = EVAL_DIR / "eval_v1.jsonl"

from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [3]:
def load_docs(data_dir: Path) -> list[dict]:
    """Return list of {source, text} dicts from all .md/.mdx files."""
    docs = []
    for path in sorted(data_dir.rglob("*.md")) + sorted(data_dir.rglob("*.mdx")):
        text = path.read_text(encoding="utf-8")
        if len(text.strip()) < 100:   # skip stub files
            continue
        docs.append({
            "source": str(path.relative_to(data_dir)),
            "text": text,
        })
    print(f"Loaded {len(docs)} documents")
    return docs

docs = load_docs(DATA_DIR)
# Preview
for d in docs[:3]:
    print(f"  {d['source']:50s}  {len(d['text'])} chars")

Loaded 9 documents
  fastapi/background-tasks.md                         4794 chars
  fastapi/bigger-applications.md                      19412 chars
  fastapi/concepts.md                                 18892 chars


In [ ]:
# Temporary chunking for Q&A Generation 

import re

def split_into_qa_chunks(text: str, max_chars: int = 3000) -> list[str]:
    """
    Split by H2 headings first; if a section is still too large, split by 
    newlines at max_chars. Returns non-empty string chunks.
    """
    # Split on markdown H2/H3 headings
    sections = re.split(r"\n(?=#{1,3} )", text)
    chunks = []
    for section in sections:
        if len(section) <= max_chars:
            if section.strip():
                chunks.append(section.strip())
        else:
            # further split by double-newline paragraphs
            parts = section.split("\n\n")
            current = ""
            for part in parts:
                if len(current) + len(part) < max_chars:
                    current += "\n\n" + part
                else:
                    if current.strip():
                        chunks.append(current.strip())
                    current = part
            if current.strip():
                chunks.append(current.strip())
    return chunks


all_chunks_for_qa = []
for doc in docs:
    chunks = split_into_qa_chunks(doc["text"], max_chars=3000)
    for c in chunks:
        all_chunks_for_qa.append({"source": doc["source"], "text": c})

print(f"Total chunks for Q&A generation: {len(all_chunks_for_qa)}")
print(f"Avg chunk length: {sum(len(c['text']) for c in all_chunks_for_qa) // len(all_chunks_for_qa)} chars")

Total chunks for Q&A generation: 140
Avg chunk length: 842 chars


In [6]:
# generating Q&A pairs for evaluation dataset

QA_PROMPT = """You are building an evaluation dataset for a RAG system about developer documentation.

Given a documentation passage below, generate {n} factual question-answer pairs.

Rules:
- Questions must be answerable ONLY from this passage (no outside knowledge)
- Questions should be diverse: some specific, some conceptual
- Answers should be 1-3 sentences, precise, and use terminology from the passage
- Format: JSON array of {{"question": "...", "answer": "...", "source_text": "first 80 chars of the relevant sentence"}}

Passage:
---
{passage}
---

Return ONLY valid JSON, no markdown fences."""


def generate_qa_pairs(passage: str, source: str, n: int = 3) -> list[dict]:
    """Call GPT-4o to generate n Q&A pairs from a passage."""
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": QA_PROMPT.format(passage=passage, n=n)}],
            temperature=0.3,
            max_tokens=800,
        )
        raw = response.choices[0].message.content.strip()
        pairs = json.loads(raw)
        for p in pairs:
            p["source"] = source
        return pairs
    except Exception as e:
        print(f"  [SKIP] {source}: {e}")
        return []
    


# Why: temperature=0.3 keeps questions factual and grounded. We ask GPT-4o to include source_text, a short quote from the passage
# This is the key: our ground truth is tied to text spans, not chunk IDs, so it survives any chunking strategy change

In [9]:
# building eval set with caching (frozen dataset)

from tqdm import tqdm

def build_eval_set(chunks: list[dict], qa_per_chunk: int = 2, chunks_per_source: int = 5) -> list[dict]:
    """
    Sample chunks_per_source chunks FROM EACH SOURCE FILE (not global limit).
    This ensures coverage across both FastAPI and Supabase docs.
    Skips if eval file already exists (frozen dataset).
    """
    if EVAL_FILE.exists():
        print(f"Eval set already exists at {EVAL_FILE} — loading it.")
        return [json.loads(l) for l in EVAL_FILE.read_text().splitlines() if l.strip()]
    
    # Group chunks by source file
    from collections import defaultdict
    by_source = defaultdict(list)
    for chunk in chunks:
        by_source[chunk["source"]].append(chunk)
    
    print(f"Found {len(by_source)} source files:")
    for src, cks in by_source.items():
        print(f"  {src}: {len(cks)} chunks")
    
    # Sample up to chunks_per_source from each file
    sampled = []
    for src, cks in by_source.items():
        sampled.extend(cks[:chunks_per_source])
    
    print(f"\nGenerating Q&A from {len(sampled)} chunks ({chunks_per_source} per file)...")
    
    all_pairs = []
    for chunk in tqdm(sampled, desc="Generating Q&A"):
        pairs = generate_qa_pairs(chunk["text"], chunk["source"], n=qa_per_chunk)
        all_pairs.extend(pairs)
    
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    with open(EVAL_FILE, "w") as f:
        for pair in all_pairs:
            f.write(json.dumps(pair) + "\n")
    
    print(f"\nSaved {len(all_pairs)} Q&A pairs to {EVAL_FILE}")
    return all_pairs

eval_set = build_eval_set(all_chunks_for_qa, qa_per_chunk=2)
print(f"Eval set size: {len(eval_set)}")
print("\nSample Q&A:")
for qa in eval_set[:2]:
    print(f"  Q: {qa['question']}")
    print(f"  A: {qa['answer'][:100]}...")
    print()

Found 9 source files:
  fastapi/background-tasks.md: 8 chunks
  fastapi/bigger-applications.md: 21 chunks
  fastapi/concepts.md: 28 chunks
  fastapi/cors.md: 8 chunks
  supabase/advanced-guide.mdx: 11 chunks
  supabase/auth-google.mdx: 22 chunks
  supabase/auth-linkedin.mdx: 11 chunks
  supabase/auth-notion.mdx: 7 chunks
  supabase/token-security.mdx: 24 chunks

Generating Q&A from 45 chunks (5 per file)...


Generating Q&A:  11%|█         | 5/45 [00:12<01:40,  2.51s/it]

  [SKIP] fastapi/background-tasks.md: Expecting value: line 1 column 1 (char 0)


Generating Q&A:  16%|█▌        | 7/45 [00:17<01:38,  2.60s/it]

  [SKIP] fastapi/bigger-applications.md: Expecting value: line 1 column 1 (char 0)


...

  [SKIP] supabase/auth-google.mdx: Expecting value: line 1 column 1 (char 0)


Generating Q&A:  84%|████████▍ | 38/45 [01:26<00:14,  2.11s/it]

  [SKIP] supabase/auth-notion.mdx: Expecting value: line 1 column 1 (char 0)


Generating Q&A: 100%|██████████| 45/45 [01:43<00:00,  2.30s/it]


Saved 78 Q&A pairs to /Users/mdayanarshad/Desktop/Switch Job UAE/kapa-inspired-rag-mcp/eval/golden_dataset/docs/eval_v1.jsonl
Eval set size: 78

Sample Q&A:
  Q: What is one example of a background task mentioned in the passage?
  A: One example of a background task is sending email notifications after performing an action....

  Q: Why might background tasks be used in a system?
  A: Background tasks are used for operations that need to happen after a request but do not require the ...

